This is the QLoRA fine-tuning process using the jsonl dataset we prepared. We'll use the Unsloth library, which significantly speeds up the training process while reducing memory requirements. This makes it perfectly suited for training on a free Google Colab Tesla T4 GPU.

#Step 1: Setup and Installation

First, we install the unsloth package along with its dependencies. Since we are using a Tesla T4 GPU on Google Colab, we will also need to install xformers (Flash Attention) to optimize memory usage.

In [ ]:
%%capture
# 1. Install Unsloth from the stable PyPI release
!pip install unsloth

# 2. Force the installation of specific compatible dependencies for Colab T4
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

#Step 2: Load the Base Model and Tokenizer

We will load the pre-trained SAWithanage/SinLlama-Llama-3-8B-Merged model. Unsloth provides a FastLanguageModel class to handle this efficiently, integrating the 4-bit quantization config.

In [ ]:
import torch
from unsloth import FastLanguageModel

# 1. Define configuration
max_seq_length = 2048 # Recommended starting length for testing
dtype = None # Auto-detects (Float16 for Tesla T4)
load_in_4bit = True # Enables 4-bit quantization (QLoRA)

# 2. Load the SinLlama base model
model_id = "SAWithanage/SinLlama-Llama-3-8B-Merged"

print(f"Loading {model_id}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Model and Tokenizer loaded successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading SAWithanage/SinLlama-Llama-3-8B-Merged...
==((====))==  Unsloth 2026.7.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load SAWithanage/SinLlama-Llama-3-8B-Merged as a legacy tokenizer.


Unsloth: SAWithanage/SinLlama-Llama-3-8B-Merged has no pad_token. Using pad_token = <|reserved_special_token_250|>.
Model and Tokenizer loaded successfully.


# Step 3: Apply LoRA Adapters
Now, we add the LoRA adapters to the model. This step is crucial because it ensures we are only training a tiny fraction of the 8 billion parameters (typically 1-10%), which is why QLoRA works on a T4 GPU.

In [ ]:
print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: Controls expressiveness vs memory (suggested: 8, 16, 32, 64)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized setting
    bias = "none",    # Optimized setting
    use_gradient_checkpointing = "unsloth", # Crucial for saving VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("Adapters configured.")

Applying LoRA adapters...


Unsloth 2026.7.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Adapters configured.


#Step 4: Load and Format and Spilt the Custom JSONL Dataset
We will load the qlora_dataset.jsonl file you generated. Unsloth's standard approach is to map the data into a specific string format before training. We will use the Alpaca template structure.

In [ ]:
from datasets import load_dataset

# 1. Load the JSONL dataset
dataset_path = "/content/qlora_dataset.jsonl"
print(f"Loading dataset from {dataset_path}...")
raw_dataset = load_dataset("json", data_files={"train": dataset_path}, split="train")

# 2. Define the Alpaca formatting template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

# 3. Apply formatting function
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []

    for instruction, input_text, output_text in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction=instruction, input=input_text, output=output_text) + EOS_TOKEN
        texts.append(text)

    return { "text" : texts }

# 4. Format dataset
formatted_dataset = raw_dataset.map(formatting_prompts_func, batched = True)

# 5. SPLIT DATASET (90% Train / 10% Test)
split_dataset = formatted_dataset.train_test_split(test_size=0.10, seed=3407)
train_dataset = split_dataset["train"]
eval_dataset  = split_dataset["test"]

print(f"✅ Total samples: {len(formatted_dataset)}")
print(f"   ├── Training samples: {len(train_dataset)}")
print(f"   └── Evaluation (Testing) samples: {len(eval_dataset)}")

Loading dataset from /content/qlora_dataset.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/126 [00:00<?, ? examples/s]

✅ Total samples: 126
   ├── Training samples: 113
   └── Evaluation (Testing) samples: 13


#Step 5: Training Configuration with SFTTrainer
Finally, we use the SFTTrainer from the trl library to execute the training loop. The parameters below are generally good starting points for small datasets.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

print("Initializing SFTTrainer with Evaluation split...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset, # Evaluation set added here
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 1,
        eval_strategy = "steps", # Automatically evaluates performance on test set
        eval_steps = 5,          # Evaluates every 5 training steps
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Start training!
print("Starting Fine-Tuning Process...")
trainer_stats = trainer.train()
print("\n--- Fine-Tuning Complete! ---")

Initializing SFTTrainer with Evaluation split...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/113 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/13 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting Fine-Tuning Process...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 113 | Num Epochs = 3 | Total steps = 45
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,162,971,648 (0.51% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
5,4.547743,4.152246
10,3.274595,3.540022
15,3.583519,3.373274
20,3.003681,3.246985
25,3.157758,3.167309
30,4.726561,3.110822
35,3.162489,3.069491
40,2.986750,3.042893
45,3.232034,3.032609


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-45/tokenizer_config.json.



--- Fine-Tuning Complete! ---


#Test the Fine-Tuned Model

In [ ]:
# 1. Switch the model to 2x faster inference mode
FastLanguageModel.for_inference(model)

# 2. Pick a noisy text from your 10% Test Split
test_noisy_text = eval_dataset[1]["input"]

# 3. Format it using the exact Alpaca prompt used during training
test_prompt = alpaca_prompt.format(
    instruction="You are a Sinhala ASR correction system. Fix the spelling and grammar of the noisy input text while preserving all numbers and context.",
    input=test_noisy_text,
    output="" # Leave output blank for the model to generate
)

# Tokenize and generate
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.1 # Keep it deterministic
)

# Print the result
generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("--- MODEL OUTPUT ---")
print(generated_text)

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- MODEL OUTPUT ---
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a Sinhala ASR correction system. Fix the spelling and grammar of the noisy input text while preserving all numbers and context.

### Input:
යිබෝවන් මරානිමඩපොලනිපුට සහය වන්නයඩුව ේමිස් අපේ පියෝටීවි එකයි තෙලිපුරනයින් එකයි දෙකම වැඩ නැහැටමීට දවස් දපලින් විල්ලත් හැුවා හැදුවත ඒ විදිහටම ආය මේ නැව  ක්තිය වෙලා තිෙනවාතෙලිෝන් එයි පියෝටීවි එකයිකනක්ෂණ එකිටකරුගෙනමෙන්ඩබ්ලි පීපී ඒ කුමාරපයිබලයින් එකක්යලින්ත්‍රයඩ් එක වැඩකරනදනෑමොකම ැඩනෑඑල්ලවසි ගෙන ලයි් කරෙඩ්වලා තින්ද බලනවෙන් වෙලා ියරෙඩ්වෙලා තයෙනවම්පෙන් එක ඇතුළත් කරන්නෑ මට සම්බලගරන් මභය නම්ම ් බින්ද හතයි හයයිරිඅසුතුනයි හඅනු එ්කයි රිටසිය පනස්පහැතිමි බැිනොනව ඳිනස නගත ස්තුතියි ම අදාලඩපන්හදනටීමකට එක ව ක්පෙන් එක දන්නෙල ලාවක්පනබය නම්බකටසෙර්ත් එකක් ගවල් මසරීමඩල ති්තූය සබ දවසසක් මෙව මැගම සඳා රැඳී සිටින්න

### Response:
ආයුබෝවන්, මම මරදාන. මට සහය වන්න. පියෝ ටීවී එ

# Evaluating the Model

In [ ]:
%%capture
!pip install evaluate jiwer rouge_score sacrebleu

In [ ]:
import torch
import evaluate
from tqdm import tqdm

# 1. Load the Evaluation Metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")
rouge_metric = evaluate.load("rouge")

# Ensure the model is in inference mode
FastLanguageModel.for_inference(model)

# 2. Lists to store predictions and true references
predictions = []
references = []

print(f"Starting evaluation on {len(eval_dataset)} test samples...")

# 3. Loop through the test set
for idx in tqdm(range(len(eval_dataset))):
    # Extract the noisy input and the true clean output from the dataset
    noisy_input = eval_dataset[idx]["input"]
    true_clean_output = eval_dataset[idx]["output"]

    # Format the prompt exactly as done during training
    test_prompt = alpaca_prompt.format(
        instruction="You are a Sinhala ASR correction system. Fix the spelling and grammar of the noisy input text while preserving all numbers and context.",
        input=noisy_input,
        output="" # Leave output blank for the model
    )

    # Tokenize and Generate
    inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

    # Generate the prediction
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode the output
    full_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Extract ONLY the generated text (strip away the prompt)
    generated_text = full_output[len(test_prompt):].strip()

    # Save the prediction and the true reference
    predictions.append(generated_text)
    references.append(true_clean_output)

# 4. Compute Metrics
print("\nComputing Metrics...")

# Compute WER and CER (Lower is better)
wer_score = wer_metric.compute(predictions=predictions, references=references)
cer_score = cer_metric.compute(predictions=predictions, references=references)

# Compute ROUGE (Higher is better)
rouge_score = rouge_metric.compute(predictions=predictions, references=references)

print("\n========================================")
print("        EVALUATION RESULTS              ")
print("========================================")
# For Error Rates, lower is better. A CER of 0 means perfect character accuracy.
print(f"Word Error Rate (WER):      {wer_score:.4f}  (Lower is better)")
print(f"Character Error Rate (CER): {cer_score:.4f}  (Lower is better)")
print("----------------------------------------")
# For ROUGE, higher is better. 1.0 means perfect overlap.
print(f"ROUGE-1 (Unigram match):    {rouge_score['rouge1']:.4f}  (Higher is better)")
print(f"ROUGE-2 (Bigram match):     {rouge_score['rouge2']:.4f}  (Higher is better)")
print(f"ROUGE-L (Sentence flow):    {rouge_score['rougeL']:.4f}  (Higher is better)")
print("========================================")

Starting evaluation on 13 test samples...


100%|██████████| 13/13 [04:33<00:00, 21.06s/it]


Computing Metrics...

        EVALUATION RESULTS              
Word Error Rate (WER):      0.9307  (Lower is better)
Character Error Rate (CER): 0.6888  (Lower is better)
----------------------------------------
ROUGE-1 (Unigram match):    0.0123  (Higher is better)
ROUGE-2 (Bigram match):     0.0000  (Higher is better)
ROUGE-L (Sentence flow):    0.0062  (Higher is better)


#Merge and Export to GGUF

In [ ]:
# Save the model locally in the Colab environment as a 4-bit quantized GGUF file
model.save_pretrained_gguf(
    "SinLlama_ASR_Cleaner",
    tokenizer,
    quantization_method = "q4_k_m"
)

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in SinLlama_ASR_Cleaner/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...



Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner`:   0%|          | 0/9 [00:00<?, ?it/s]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner`:  11%|█         | 1/9 [00:17<02:21, 17.69s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner`:  22%|██▏       | 2/9 [00:58<03:39, 31.36s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner`:  33%|███▎      | 3/9 [01:46<03:52, 38.78s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner`:  44%|████▍     | 4/9 [02:32<03:28, 41.78s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner`:  56%|█████▌    | 5/9 [03:16<02:49, 42.45s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner`:  67%|██████▋   | 6/9 [04:03<02:11, 44.00s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner`:  78%|███████▊  | 7/9 [04:54<01:32, 46.40s/it]

Unsloth: Copying 9 files from cache to `SinLlama_ASR_Cleaner`:  89%|████████▉ | 8/9 [05:49<00:49, 49.08s/it]

Unsloth: Copying

Successfully copied all 9 files from cache to `SinLlama_ASR_Cleaner`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.




Unsloth: Preparing safetensor model files: 100%|██████████| 9/9 [00:00<00:00, 55027.31it/s]


Unsloth: Merging weights into 16bit:   0%|          | 0/9 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  11%|█         | 1/9 [00:19<02:36, 19.58s/it]

Unsloth: Merging weights into 16bit:  22%|██▏       | 2/9 [01:06<04:08, 35.45s/it]

Unsloth: Merging weights into 16bit:  33%|███▎      | 3/9 [02:00<04:24, 44.16s/it]

Unsloth: Merging weights into 16bit:  44%|████▍     | 4/9 [02:56<04:03, 48.72s/it]

Unsloth: Merging weights into 16bit:  56%|█████▌    | 5/9 [03:51<03:23, 50.95s/it]

Unsloth: Merging weights into 16bit:  67%|██████▋   | 6/9 [04:49<02:40, 53.49s/it]

Unsloth: Merging weights into 16bit:  78%|███████▊  | 7/9 [05:40<01:45, 52.63s/it]

Unsloth: Merging weights into 16bit:  89%|████████▉ | 8/9 [06:30<00:51, 51.79s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 9/9 [07:06<00:00, 47.41s/it]


Unsloth: Merge process complete. Saved to `/content/SinLlama_ASR_Cleaner`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10194-mix-f08678f (app-b10194-mix-f08678f-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['SinLlama_ASR_Cleaner_gguf/SinLlama-Llama-3-8B-Merged.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conve

{'save_directory': 'SinLlama_ASR_Cleaner',
 'gguf_directory': 'SinLlama_ASR_Cleaner_gguf',
 'gguf_files': ['SinLlama_ASR_Cleaner_gguf/SinLlama-Llama-3-8B-Merged.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}